# ★ Wow · a live 3-D view

The bonus notebook draws a still point cloud. Here we **redraw the gog4j plot a
couple of times a second** with freshly propagated positions, so the fleet
appears to orbit the Earth.

The trick: `display(...)` the container **once**, then swap a new plot into it on
every tick. The notebook keeps that node in the live scene graph, so the swap
shows.

In [0]:
addDependency("org.orekit:orekit:13.0.3");
addDependency("org.dflib:dflib:2.0.0-M7");
addDependency("org.dflib:dflib-parquet:2.0.0-M7");
addDependency("org.jtaccuino:gog4j:0.5-SNAPSHOT");
addDependency("org.jtaccuino:gog4j-hardwood:0.5-SNAPSHOT");

// the jars above are on the class path for the snippets that follow; name any
// that did not land, rather than failing later with "cannot find symbol"
try { Class.forName("org.orekit.data.DataContext"); }
catch (ClassNotFoundException notThere) {
    println("warning: org.orekit.data.DataContext is not on the class path yet (%s)",
            notThere.getMessage());
}

import org.dflib.DataFrame;
import org.dflib.parquet.Parquet;
import org.orekit.data.DataContext;
import org.orekit.time.AbsoluteDate;
import org.orekit.propagation.analytical.tle.TLE;
import org.orekit.propagation.analytical.tle.TLEPropagator;
import org.jtaccuino.gog.*;
import org.jtaccuino.gog.labs.Labs;
import org.jtaccuino.gog.render.SvgExporter;
import org.jtaccuino.gog.theme.Theme;
import org.jtaccuino.gog.hardwood.HardwoodDataFrame;
import java.time.Instant;
import java.util.ArrayList;
import java.util.LinkedHashMap;
import java.util.List;
import java.util.Map;
import java.util.concurrent.atomic.AtomicInteger;
import java.nio.file.Files;
import java.nio.file.Path;
import javafx.animation.AnimationTimer;
import javafx.scene.layout.StackPane;
import static org.dflib.Exp.*;

String num(long v) { return String.format(java.util.Locale.ROOT, "%,d", v); }
String dec(double v, int p) { return String.format(java.util.Locale.ROOT, "%." + p + "f", v); }

// The notebook may live in a sub-folder (exercises/, bonus/, ...), so walk up
// from the notebook's own folder to find the repository's data/ directory.
Path dataset() {
    for (Path p = cwd.toAbsolutePath(); p != null; p = p.getParent()) {
        Path candidate = p.resolve("data/celestrak_gp_catalog.parquet");
        if (Files.isRegularFile(candidate)) return candidate;
    }
    throw new IllegalStateException("data/celestrak_gp_catalog.parquet not found above " + cwd);
}

var df = Parquet.loader().load(dataset());
var tai = DataContext.getDefault().getTimeScales().getTAI();
var now = new AbsoluteDate(Instant.now(), tai);

// Everything that touches JavaFX — building the plot's canvas, exporting,
// attaching a node — must run on the FX Application Thread. Notebook code runs
// on JShell's worker thread, so marshal with onFx(...) and wait for it.
void onFx(Runnable work) {
    if (javafx.application.Platform.isFxApplicationThread()) {
        work.run();
        return;
    }
    var latch = new java.util.concurrent.CountDownLatch(1);
    javafx.application.Platform.runLater(() -> {
        try { work.run(); } catch (Throwable t) { t.printStackTrace(); } finally { latch.countDown(); }
    });
    try { latch.await(); } catch (InterruptedException e) { Thread.currentThread().interrupt(); }
}

void save(GgFigure figure, String name) {
    var path = cwd.resolve(name);
    var size = new long[]{-1};
    var error = new String[]{null};
    onFx(() -> {
        try {
            new SvgExporter().size(1200, 900).batchPoints(true).write(figure, path);
            size[0] = Files.size(path);
        } catch (Exception e) {
            error[0] = String.valueOf(e);
        }
    });
    if (error[0] != null) {
        println("could not save %s: %s", name, error[0]);
    } else {
        println("saved %s (%s bytes)", name, num(size[0]));
    }
}

## 1 · A fleet and its frames

Take a few hundred **LEO** objects and work out where each one is at a series of
instants — one instant per frame. Precomputing the frames keeps the drawing loop
cheap; the loop itself only re-renders.

In [0]:
// ── given ── the fleet, and one position per object per frame
var leo = df.rows($str("orbit_class").eq("LEO")).select();
var leoL1 = leo.getColumn("tle_line1");
var leoL2 = leo.getColumn("tle_line2");
var leoClass = leo.getColumn("orbit_class");

int fleetSize = 500;          // objects drawn at once
int frames = 60;              // one lap of the flip-book
double secondsPerFrame = 90;  // real time advanced per frame

var fleetL1 = new ArrayList<String>();
var fleetL2 = new ArrayList<String>();
var fleetClass = new ArrayList<String>();
for (int i = 0; i < leo.height() && fleetL1.size() < fleetSize; i++) {
    Object a = leoL1.get(i), b = leoL2.get(i);
    if (a != null && b != null) {
        fleetL1.add(a.toString());
        fleetL2.add(b.toString());
        fleetClass.add(String.valueOf(leoClass.get(i)));
    }
}

// positions[frame][object] = x, y, z in km
double[][][] positions = new double[frames][fleetL1.size()][3];
for (int f = 0; f < frames; f++) {
    var at = now.shiftedBy(f * secondsPerFrame);
    for (int i = 0; i < fleetL1.size(); i++) {
        try {
            var tle = new TLE(fleetL1.get(i), fleetL2.get(i), tai);
            var pos = TLEPropagator.selectExtrapolator(tle).propagate(at)
                        .getPVCoordinates().getPosition();
            positions[f][i][0] = pos.getX() / 1000.0;
            positions[f][i][1] = pos.getY() / 1000.0;
            positions[f][i][2] = pos.getZ() / 1000.0;
        } catch (Exception e) {
            // this object will not propagate - leave it at the origin
        }
    }
}
println("precomputed %d frames for %s objects", frames, num(fleetL1.size()));

## 2 · Redraw the plot, twice a second

Build a gog4j 3-D plot for one frame, and swap it into a container we displayed
once. An `AnimationTimer` throttles itself to ~2 fps — enough to read as motion,
cheap enough to re-render a few hundred points.

In [0]:
// ── given ── a fresh plot for one frame
java.util.function.IntFunction<Plot<HardwoodDataFrame>> plotFor = f -> {
    var xs = new ArrayList<Double>();
    var ys = new ArrayList<Double>();
    var zs = new ArrayList<Double>();
    for (int i = 0; i < fleetL1.size(); i++) {
        xs.add(positions[f][i][0]);
        ys.add(positions[f][i][1]);
        zs.add(positions[f][i][2]);
    }
    var cols = new LinkedHashMap<String, List<?>>();
    cols.put("x", xs);
    cols.put("y", ys);
    cols.put("z", zs);
    cols.put("orbit_class", fleetClass);
    var table = HardwoodDataFrame.ofColumns(cols);
    var plot = Ggplot.ggplot3d(table, Aes.aes().x("x").y("y").z("z").color("orbit_class"))
        .geoms(Geoms.point3d())
        .labs(Labs.labs("LEO, live", "x (km)", "y (km)"))
        .theme(Theme.theme_dark());
    plot.setPrefSize(780, 600);
    return plot;
};
println("plotFor is ready");

In [0]:
// ── given ── display once, then swap the plot on every tick
var stage = new StackPane();
stage.setPrefSize(780, 600);
display(stage);

// the swap touches the live scene graph, so it must happen on the FX thread —
// and so must starting the timer. From then on every tick already runs there.
onFx(() -> {
    stage.getChildren().setAll(plotFor.apply(0));
    var frameNo = new AtomicInteger();
    var timer = new AnimationTimer() {
        private long last;
        @Override
        public void handle(long nanos) {
            if (nanos - last < 500_000_000L) return;   // ~ 2 frames per second
            last = nanos;
            int f = frameNo.getAndIncrement() % frames;
            stage.getChildren().setAll(plotFor.apply(f));
        }
    };
    timer.start();
});
println("redrawing the plot twice a second — %s points per frame", num(fleetL1.size()));

In [0]:
// ── given ── one frame as a file, for the record
save(plotFor.apply(0), "orbits-live-frame.svg");

In [0]:
// ── check ─────────────────────────────────────────────────────────────
var pass = fleetL1.size() > 100 && frames > 10 && positions.length == frames;
if (pass) {
    println("✓ %s points × %d frames — redrawn live at ~2 fps", num(fleetL1.size()), frames);
} else {
    println("✗ something is off — check the setup above");
}

Reload the notebook and run it again tomorrow: the fleet is somewhere else, and
the animation follows the *current* element sets.

It is the same pipeline as the other notebooks — **Hardwood** reads, **dflib**
shapes, **Orekit** propagates, **gog4j** draws — just on a clock.

Back to the [overview](../narrative/00-overview.md).